# 03 · Custom collate functions

`CustomCollate` does the padding and masking. **What the model learns is decided by your collate
functions**: one per feature, passed as `funcs={name: function}`. Each one says what the input is
and what the target is.

| Pattern | Task | Target | Section |
|---|---|---|---|
| next token | classification | the next id | 3A |
| next value | regression | the next (scaled) value | 3B |
| **delta** | regression | the change since the last step | 3C |
| **delta bucket** | classification | the change, as a class (down / flat / up) | 3D |
| ratio | regression | `log(next / last)`, scale-free | 3E |
| scaled input, raw target | either | input and target from different versions of the value | 3F |
| several targets from one column | both | one feature key per target | 3G |

Sections 4–5 cover checking a function and a common bug; section 6 trains a model with a delta
target and a delta-bucket target and turns the predictions back into prices.

## 1. The contract

`CustomCollate` calls `funcs[name](values, inference)` once per batch and feature:

- `values`: that feature's value from **every sample in the batch** (a list of histories). A
  function only sees its own feature.
- returns `(inputs, targets)`:
  - `inputs`: one 1-D tensor per sample. `long` ids for categorical features, `float` for numeric (`is_linear=True`).
  - `targets`: one scalar per sample; `None` when `inference=True`.
- **training** (`inference=False`): the last step is the event to predict, so inputs are every step but the last.
- **inference** (`inference=True`): inputs are the full history and there is no target.

After your function, `CustomCollate` truncates to `max_len` (keeping the newest steps), pads with 0
and builds the mask. Watch it being called:

In [1]:
import math
import torch
from next_sequence import CustomCollate, Feature, TaskType

def collate_spy(values, inference=False):
    print(f"called with inference={inference}, values={values}")
    xs = [torch.tensor(v, dtype=torch.float) for v in values]
    return (xs, None) if inference else ([x[:-1] for x in xs], [v[-1] for v in values])

spy_collate = CustomCollate([Feature("price", "regression", is_linear=True)], "price", {"price": collate_spy})
batch = spy_collate([{"price": [10.0, 12.0, 15.0]}, {"price": [20.0, 18.0]}])
print("X:", batch["X"]["price"].squeeze(-1).tolist(), "| Y:", batch["Y"]["price"].tolist())

called with inference=False, values=[[10.0, 12.0, 15.0], [20.0, 18.0]]
X: [[10.0, 12.0], [20.0, 0.0]] | Y: [15.0, 18.0]


## 2. One helper for every collate function

Writing the training and inference branches by hand for every feature is how they drift apart
(section 5 shows that bug). `make_collate` builds both from the same two small functions:

- `x(step)`: the input value of one step (default: the step itself)
- `y(history)`: the target, computed from the **whole training history**; `history[-1]` is the event
  to predict, `history[-2]` the last one the model sees

In [2]:
def make_collate(dtype, x=lambda step: step, y=lambda history: history[-1]):
    """Build a collate function whose training and inference branches can't disagree."""
    def collate(values, inference=False):
        xs = [torch.tensor([x(step) for step in history], dtype=dtype) for history in values]
        if inference:
            return xs, None                                   # full history, no target
        return [t[:-1] for t in xs], [y(history) for history in values]
    return collate

toy = [[10.0, 12.0, 15.0], [20.0, 18.0]]   # two price histories

## 3. Patterns

### 3A. Next token → classification
Inputs and target are ids from the same vocabulary. `Feature(..., "classification", vocab_size=len(bow))`.

In [3]:
collate_next_token = make_collate(torch.long)
collate_next_token([[2, 3, 4], [5, 6]])

([tensor([2, 3]), tensor([5])], [4, 6])

### 3B. Next value → regression
Feed **scaled** values (fitted on the training split). The prediction comes back in the same scale.
`Feature(..., "regression", is_linear=True)`.

In [4]:
mean, std = 15.0, 4.0                                     # fitted on the training split
collate_next_value = make_collate(torch.float, x=lambda p: (p - mean) / std, y=lambda h: (h[-1] - mean) / std)
collate_next_value(toy)

([tensor([-1.2500, -0.7500]), tensor([1.2500])], [0.0, 0.75])

### 3C. Delta → regression

Predict the **change** since the last step instead of the value itself. Changes are often easier to
learn, and they keep the model honest about "same as before".

Scale the target by the standard deviation of the changes **in the training data**; raw changes
(10, 500, 1500, ...) give a huge MSE that drowns every other task.

**Decoding:** `predicted value = last known value + predicted delta * delta_std`.

In [5]:
delta_std = 2.5                                           # std of training deltas
collate_delta = make_collate(torch.float, x=lambda p: (p - mean) / std,
                             y=lambda h: (h[-1] - h[-2]) / delta_std)
x, y = collate_delta(toy)
print("inputs :", [t.tolist() for t in x])
print("targets:", y, " <- (15-12)/2.5 and (18-20)/2.5")

inputs : [[-1.25, -0.75], [1.25]]
targets: [1.2, -0.8]  <- (15-12)/2.5 and (18-20)/2.5


A delta needs at least **two** steps in a training history (`history[-2]`). Drop shorter samples from
the training data, or return a fallback such as `0.0`.

### 3D. Delta bucket → classification

The same change, as classes. **Classes start at 0**; set `n_classes` to how many there are.
`Feature(..., "classification", is_linear=True, n_classes=3)` for a numeric input with a class target.

In [6]:
MOVES = ["down", "flat", "up"]                              # class 0, 1, 2

def move_bucket(delta, flat=1.0):
    return 0 if delta < -flat else (1 if delta <= flat else 2)

collate_delta_bucket = make_collate(torch.float, x=lambda p: (p - mean) / std,
                                    y=lambda h: move_bucket(h[-1] - h[-2]))
x, y = collate_delta_bucket(toy)
print("targets:", y, "->", [MOVES[c] for c in y])

targets: [2, 0] -> ['up', 'down']


### 3E. Ratio → regression

`log(next / last)` is scale-free: a 10% rise is the same target for a 20 and a 2000 price.
**Decoding:** `predicted value = last known value * exp(prediction)`.

In [7]:
collate_log_ratio = make_collate(torch.float, x=lambda p: (p - mean) / std,
                                 y=lambda h: math.log(h[-1] / h[-2]))
collate_log_ratio(toy)[1]

[0.22314355131420976, -0.10536051565782628]

### 3F. Scaled input, raw target

Sometimes the target needs the raw value (a delta in real units, a threshold in dollars) while the
input should be scaled. A collate function only sees its own feature, so store **both** per step in
the sample, e.g. `(scaled, raw)`, and let `x` and `y` pick:

In [8]:
steps = [[((p - mean) / std, p) for p in history] for history in toy]   # (scaled, raw) per step
collate_scaled_in_raw_out = make_collate(torch.float, x=lambda s: s[0],
                                         y=lambda h: move_bucket(h[-1][1] - h[-2][1], flat=1.0))
x, y = collate_scaled_in_raw_out(steps)
print("inputs (scaled):", [t.tolist() for t in x], "| targets from raw values:", y)

inputs (scaled): [[-1.25, -0.75], [1.25]] | targets from raw values: [2, 0]


### 3G. Several targets from one column

Each feature has exactly one target. To predict both the price change **and** its direction, give
the sample two keys built from the same column and declare two features:

```python
sample = {"price": steps, "price_move": steps, ...}
features = [Feature("price", "regression", is_linear=True),                    # delta
            Feature("price_move", "classification", is_linear=True, n_classes=3)]   # delta bucket
funcs = {"price": collate_delta, "price_move": collate_delta_bucket}
```

Both are also inputs, so the model sees the price twice. That's harmless. Section 6 uses exactly this.

## 4. Check a collate function before training

A few lines catch most mistakes: the branches must agree on dtype, training must drop exactly the
last step, and there must be one target per sample.

In [9]:
def check_collate(collate, values):
    x_train, y_train = collate(values, False)
    x_infer, y_infer = collate(values, True)
    assert y_infer is None, "inference must return targets=None"
    assert len(y_train) == len(values), "need one target per sample"
    for a, b in zip(x_train, x_infer):
        assert a.dtype == b.dtype, f"training inputs are {a.dtype} but inference inputs are {b.dtype}"
        assert len(a) == len(b) - 1, "training inputs should drop exactly the last step"
    print(f"ok | inputs {x_train[0].dtype} | targets {y_train}")

for name, fn in [("next_value", collate_next_value), ("delta", collate_delta), ("delta_bucket", collate_delta_bucket)]:
    print(f"{name:13s}", end=" ")
    check_collate(fn, toy)

next_value    ok | inputs torch.float32 | targets [0.0, 0.75]
delta         ok | inputs torch.float32 | targets [1.2, -0.8]
delta_bucket  ok | inputs torch.float32 | targets [2, 0]


## 5. A common bug: branches that disagree

Written by hand, it's easy for the inference branch to build different inputs than the training
branch. Here inference makes `long` inputs and training makes `float` ones, and the classes start at 1:

In [10]:
def collate_delta_bin_handwritten(batch, inference=False):
    if inference:
        return [torch.tensor(b, dtype=torch.long) for b in batch], None   # long ...
    def logic(d):
        return 1 if d <= 1 else (2 if d <= 3 else 3)                       # classes 1..3, class 0 unused
    x = [torch.tensor(b[:-1], dtype=torch.float) for b in batch]           # ... but float here
    y = [logic(b[-1] - b[-2]) for b in batch]
    return x, y

try:
    check_collate(collate_delta_bin_handwritten, [[25, 26, 27], [35, 37]])
except AssertionError as e:
    print("caught:", e)

caught: training inputs are torch.float32 but inference inputs are torch.int64


The `make_collate` version can't have this bug, and its classes start at 0 (`n_classes=3`):

In [11]:
def age_bucket(d):
    return 0 if d <= 1 else (1 if d <= 3 else 2)

collate_delta_bin = make_collate(torch.float, y=lambda h: age_bucket(h[-1] - h[-2]))
check_collate(collate_delta_bin, [[25, 26, 27], [35, 37]])

ok | inputs torch.float32 | targets [0, 1]


## 6. End to end: next item + price delta + price direction

Same shop data as the other cookbooks. Three things are predicted at once:

- `item`: next item (3A)
- `price`: the **change** in price, scaled by the training delta std (3C)
- `price_move`: the direction of that change: down / flat / up (3D)

Both price features use `(scaled log-price, raw price)` steps, so the input is scaled and the targets
use real prices (3F, 3G).

## The example data

Every cookbook uses the same small synthetic shop: each row is one purchase with an `item`,
a `price` and a `weekday`. Shoppers buy the five items in a fixed cycle
(`apple → bread → cheese → dates → eggs → apple ...`), starting at a random item, so the next
item is learnable and we can check the model gets it right.

In [12]:
import random
import pandas as pd

ITEMS = ["apple", "bread", "cheese", "dates", "eggs"]
PRICE = {"apple": 20.0, "bread": 45.0, "cheese": 180.0, "dates": 95.0, "eggs": 60.0}

def make_events(n_users=300, seed=0):
    """One row per purchase. Each shopper buys ITEMS in a fixed cycle, starting at a random item."""
    rng = random.Random(seed)
    rows = []
    for user in range(n_users):
        start = rng.randrange(len(ITEMS))
        for t in range(rng.randint(6, 12)):
            item = ITEMS[(start + t) % len(ITEMS)]
            rows.append({"user_id": user, "t": t, "item": item,
                         "price": round(PRICE[item] * rng.uniform(0.9, 1.1), 2),
                         "weekday": rng.randrange(7)})
    return pd.DataFrame(rows)

from torch.utils.data import DataLoader
from next_sequence import CoreDataset, NextSequence, NextSequenceConfig, create_bow

events = make_events()
users = (events.sort_values(["user_id", "t"]).groupby("user_id").agg(list)
               .reset_index().drop(columns="t").to_dict(orient="records"))

def cut(user, end):
    return {k: (v[:end] if isinstance(v, list) else v) for k, v in user.items()}

train_raw = [cut(u, -2) for u in users]
val_raw = [cut(u, -1) for u in users]
test_raw = [cut(u, None) for u in users]

Fit everything on the training split: the vocabulary, the input scaler and the **delta** scaler.

In [13]:
bow = create_bow(u["item"] for u in train_raw)
log_prices = [math.log1p(p) for u in train_raw for p in u["price"]]
price_mean = sum(log_prices) / len(log_prices)
price_std = (sum((v - price_mean) ** 2 for v in log_prices) / len(log_prices)) ** 0.5

deltas = [b - a for u in train_raw for a, b in zip(u["price"][:-1], u["price"][1:])]
delta_std = (sum(d * d for d in deltas) / len(deltas)) ** 0.5
FLAT = 30.0                                             # |change| <= 30 counts as flat
counts = [sum(move_bucket(d, FLAT) == c for d in deltas) for c in range(3)]
print(f"delta std: {delta_std:.1f} | training moves: {dict(zip(MOVES, counts))}")

delta std: 76.1 | training moves: {'down': 1001, 'flat': 420, 'up': 355}


In [14]:
def transform(samples):
    out = []
    for u in samples:
        steps = [((math.log1p(p) - price_mean) / price_std, p) for p in u["price"]]   # (scaled, raw)
        out.append({"item": [bow.get(i, bow["UNK"]) for i in u["item"]],
                    "price": steps, "price_move": steps,
                    "weekday": [d + 1 for d in u["weekday"]]})
    return out

train, val, test = transform(train_raw), transform(val_raw), transform(test_raw)

features = [
    Feature("item", TaskType.CLASSIFICATION, vocab_size=len(bow)),
    Feature("price", TaskType.REGRESSION, is_linear=True),                        # delta
    Feature("price_move", TaskType.CLASSIFICATION, is_linear=True, n_classes=3),  # delta bucket
    Feature("weekday", TaskType.PASSTHROUGH, vocab_size=8),
]
funcs = {
    "item": make_collate(torch.long),
    "price": make_collate(torch.float, x=lambda s: s[0], y=lambda h: (h[-1][1] - h[-2][1]) / delta_std),
    "price_move": make_collate(torch.float, x=lambda s: s[0], y=lambda h: move_bucket(h[-1][1] - h[-2][1], FLAT)),
    "weekday": make_collate(torch.long),
}
for name in ("price", "price_move"):
    print(f"{name:10s}", end=" ")
    check_collate(funcs[name], [s[name] for s in train[:3]])

price      ok | inputs torch.float32 | targets [1.944343331294591, -0.37417999915717237, 0.37812150195728306]
price_move ok | inputs torch.float32 | targets [2, 1, 1]


In [15]:
MAX_LEN = 16

def loader(samples, inference=False, shuffle=False):
    collate = CustomCollate(features, "item", funcs, inference=inference, max_len=MAX_LEN)
    return DataLoader(CoreDataset(samples), batch_size=32, shuffle=shuffle, collate_fn=collate)

config = NextSequenceConfig(features=features, max_len=MAX_LEN, emb_dim=32, attn_heads=2)
model = NextSequence(config, epochs=20, lr=3e-3, seed=0, verbose=False)
model.fit(loader(train, shuffle=True), loader(val))
pd.DataFrame(model.history_).set_index("epoch").iloc[[0, 4, 9, 14, 19]].round(4)

,train_loss,train_item,train_price,train_price_move,val_loss,val_item,val_price,val_price_move
epoch,,,,,,,,
1,3.4315,1.8659,1.1765,0.9846,3.1001,1.6523,0.9647,0.9913
5,1.2790,0.8132,0.2475,0.3833,1.1712,0.7737,0.1765,0.3577
10,0.1479,0.0548,0.0479,0.1369,0.1860,0.0455,0.0350,0.2019
15,-0.0352,0.0225,0.0278,0.0791,0.0826,0.0171,0.0217,0.1991
20,-0.0825,0.0196,0.0225,0.0573,0.0859,0.0176,0.0219,0.2114


### Decode the predictions back into prices

On the test split the model saw every step but the last, so the "last known" price is `price[-2]`:

- price: `last known + predicted delta * delta_std`
- direction: `argmax` over the three classes

In [16]:
out = model.predict(loader(test))
last_known = torch.tensor([u["price"][-2] for u in test_raw])
true_price = torch.tensor([u["price"][-1] for u in test_raw])
pred_price = last_known + out["price"].squeeze(-1) * delta_std

true_move = [move_bucket(b - a, FLAT) for a, b in zip(last_known.tolist(), true_price.tolist())]
pred_move = out["price_move"].argmax(-1).tolist()

baseline = (last_known - true_price).abs().mean().item()                # "price stays the same"
print(f"price MAE: {(pred_price - true_price).abs().mean():.2f}  (repeat-last-price baseline: {baseline:.2f})")
print(f"direction accuracy: {sum(p == t for p, t in zip(pred_move, true_move)) / len(true_move):.3f}")

pd.DataFrame({
    "item": [f"{u['item'][-2]} → {u['item'][-1]}" for u in test_raw[:6]],
    "last price": last_known[:6].double().numpy().round(2),
    "true price": true_price[:6].double().numpy().round(2),
    "predicted": pred_price[:6].double().numpy().round(2),
    "true move": [MOVES[c] for c in true_move[:6]],
    "predicted move": [MOVES[c] for c in pred_move[:6]],
})

price MAE: 12.58  (repeat-last-price baseline: 67.20)
direction accuracy: 0.920


,item,last price,true price,predicted,true move,predicted move
0,dates → eggs,96.94,57.72,26.57,down,down
1,apple → bread,21.14,40.51,-4.38,flat,flat
2,cheese → dates,165.28,102.13,77.18,down,down
3,bread → cheese,45.87,173.41,188.07,up,up
4,cheese → dates,195.24,87.02,109.54,down,down
5,dates → eggs,97.14,61.04,63.19,down,down


### Predicting the future

With `inference=True` the model sees the full history, so the last known price is `price[-1]`:

In [17]:
future = model.predict(loader(test, inference=True))
now = torch.tensor([u["price"][-1] for u in test_raw])
pd.DataFrame({
    "last item": [u["item"][-1] for u in test_raw[:5]],
    "current price": now[:5].double().numpy().round(2),
    "predicted next price": (now + future["price"].squeeze(-1) * delta_std)[:5].double().numpy().round(2),
    "predicted move": [MOVES[c] for c in future["price_move"].argmax(-1)[:5].tolist()],
})

,last item,current price,predicted next price,predicted move
0,eggs,57.72,-18.76,down
1,bread,40.51,73.26,flat
2,dates,102.13,63.68,down
3,cheese,173.41,91.62,down
4,dates,87.02,44.28,down


### Delta or ratio? Look at the predicted prices

A delta adds a fixed amount regardless of the current level, so a predicted drop of 25 on a price of
21 goes **below zero**. For quantities that are always positive (prices, durations, counts), a ratio
target (3E) can't: `last * exp(prediction)` is always positive. Same model, only the `price` collate
function changes:

In [18]:
log_ratios = [math.log(b / a) for u in train_raw for a, b in zip(u["price"][:-1], u["price"][1:])]
ratio_std = (sum(r * r for r in log_ratios) / len(log_ratios)) ** 0.5

funcs_ratio = {**funcs, "price": make_collate(torch.float, x=lambda s: s[0],
                                              y=lambda h: math.log(h[-1][1] / h[-2][1]) / ratio_std)}

def ratio_loader(samples, inference=False, shuffle=False):
    collate = CustomCollate(features, "item", funcs_ratio, inference=inference, max_len=MAX_LEN)
    return DataLoader(CoreDataset(samples), batch_size=32, shuffle=shuffle, collate_fn=collate)

ratio_model = NextSequence(config, epochs=20, lr=3e-3, seed=0, verbose=False)
ratio_model.fit(ratio_loader(train, shuffle=True), ratio_loader(val))
ratio_price = last_known * torch.exp(ratio_model.predict(ratio_loader(test))["price"].squeeze(-1) * ratio_std)

pd.DataFrame({
    "target": ["delta", "log ratio"],
    "price MAE": [(pred_price - true_price).abs().mean().item(), (ratio_price - true_price).abs().mean().item()],
    "lowest predicted price": [pred_price.min().item(), ratio_price.min().item()],
    "negative predictions": [int((pred_price < 0).sum()), int((ratio_price < 0).sum())],
}).round(2)

,target,price MAE,lowest predicted price,negative predictions
0,delta,12.58,-6.78,4
1,log ratio,15.22,10.89,0


It's a trade-off, not a free win: on this data the delta target has the lower average error, but it
predicts some impossible negative prices; the ratio target never does. Pick the target by what matters
for your use (plausible values vs. the lowest average error), and compare both on your validation split.

## Summary

| You want to predict | Collate function | Feature | Decode |
|---|---|---|---|
| next token | `make_collate(torch.long)` | classification, `vocab_size` | `argmax` → id → token |
| next value | `y=lambda h: scale(h[-1])` | regression, `is_linear=True` | unscale |
| change (delta) | `y=lambda h: (h[-1] - h[-2]) / delta_std` | regression, `is_linear=True` | `last + pred * delta_std` |
| change as classes | `y=lambda h: bucket(h[-1] - h[-2])` | classification, `is_linear=True, n_classes=k` | `argmax` → label |
| ratio (positive values) | `y=lambda h: log(h[-1] / h[-2]) / ratio_std` | regression, `is_linear=True` | `last * exp(pred * ratio_std)` |

Rules that always apply:
- one helper builds both branches; check new functions with `check_collate`
- fit every scaler (values **and** deltas) on the training split only
- classes start at 0
- a delta or ratio target needs at least two steps of history
- a delta can push a positive quantity below zero; use a ratio for prices, durations and counts